<a href="https://colab.research.google.com/github/sachitha-m-k/Statistical-Learning-e23168/blob/main/data_wrangling_assignment_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine


## Step 1 — Install & Import Dependencies

In [ ]:
# Install any libraries not available by default in Colab
!pip install -q plotly pandas numpy scikit-learn scipy

In [ ]:
import io
import math
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pointbiserialr, f_oneway

# Scikit-learn
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler,
    OrdinalEncoder, OneHotEncoder
)

# Plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Colab file upload
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings('ignore')
print('All dependencies loaded successfully.')

## Step 2 — `PlottingMethods` Class (Modular Charts)

In [ ]:
class PlottingMethods:
    """
    A standalone utility class providing granular chart-generation methods.
    Every method returns an HTML string wrapping a Plotly figure so the
    output can be embedded anywhere (Colab cell, Flask page, report, etc.).
    """

    @staticmethod
    def _to_html(fig: go.Figure) -> str:
        """Convert a Plotly figure to a self-contained HTML string."""
        return fig.to_html(full_html=False, include_plotlyjs='cdn')

    @staticmethod
    def bar_chart(
        categories,
        values,
        title: str = 'Bar Chart',
        x_label: str = 'Category',
        y_label: str = 'Value',
        color: str = '#636EFA',
        show: bool = True
    ) -> str:
        """
        Generate a vertical bar chart.

        Parameters
        ----------
        categories : array-like  — x-axis labels.
        values     : array-like  — bar heights.
        title      : str         — chart title.
        x_label    : str         — x-axis label.
        y_label    : str         — y-axis label.
        color      : str         — bar fill colour (hex / named).
        show       : bool        — if True, call fig.show() in-notebook.

        Returns
        -------
        str : HTML string of the rendered figure.
        """
        if categories is None or values is None:
            print('bar_chart: categories and values must not be None.')
            return ''
        fig = go.Figure(
            go.Bar(x=list(categories), y=list(values),
                   marker_color=color, text=list(values),
                   textposition='outside')
        )
        fig.update_layout(
            title=title,
            xaxis_title=x_label,
            yaxis_title=y_label,
            template='plotly_white'
        )
        if show:
            fig.show()
        return PlottingMethods._to_html(fig)

    @staticmethod
    def pie_chart(
        labels,
        values,
        title: str = 'Pie Chart',
        show: bool = True
    ) -> str:
        """
        Generate a pie chart with percentage labels.

        Parameters
        ----------
        labels : array-like — slice labels.
        values : array-like — slice sizes.
        title  : str        — chart title.
        show   : bool       — if True, call fig.show().

        Returns
        -------
        str : HTML string.
        """
        if labels is None or values is None:
            print('pie_chart: labels and values must not be None.')
            return ''
        fig = go.Figure(
            go.Pie(labels=list(labels), values=list(values),
                   textinfo='label+percent', hole=0.3)
        )
        fig.update_layout(title=title, template='plotly_white')
        if show:
            fig.show()
        return PlottingMethods._to_html(fig)

    @staticmethod
    def histogram(
        data,
        column: str,
        nbins: int = 30,
        title: str = None,
        color: str = '#EF553B',
        show: bool = True
    ) -> str:
        """
        Generate a histogram for a single numeric column.

        Parameters
        ----------
        data   : pd.DataFrame — source DataFrame.
        column : str          — column name to plot.
        nbins  : int          — number of histogram bins.
        title  : str          — chart title (auto-generated if None).
        color  : str          — bar fill colour.
        show   : bool         — if True, call fig.show().

        Returns
        -------
        str : HTML string.
        """
        if data is None or data.empty or column not in data.columns:
            print(f'histogram: column "{column}" not found or data is empty.')
            return ''
        title = title or f'Histogram — {column}'
        fig = px.histogram(
            data, x=column, nbins=nbins, title=title,
            color_discrete_sequence=[color],
            template='plotly_white'
        )
        fig.update_layout(bargap=0.05)
        if show:
            fig.show()
        return PlottingMethods._to_html(fig)

print('PlottingMethods class defined.')

## Step 3 — `DataInspector` Class

In [ ]:
class DataInspector:
    """
    A reusable, object-oriented tool for CSV data ingestion, cleaning,
    feature engineering preparation, and statistical visualisation.

    Attributes
    ----------
    df : pd.DataFrame
        The working dataset. All mutating methods operate on this attribute
        in-place so that operations chain naturally.
    _original_df : pd.DataFrame
        Snapshot of the DataFrame as first loaded; used for reset operations.
    """

    # ── Strings that should be treated as missing
    GARBAGE_STRINGS = [
        '?', 'n/a', 'N/A', 'na', 'NA', 'null', 'NULL', 'Null',
        'none', 'None', 'NONE', '', ' ', '--', '---', 'nan', 'NaN'
    ]

    # 1 ─ DATA INGESTION & SANITISATION
    ###############################################

    def __init__(self, df: pd.DataFrame = None):
        """
        Initialise with an optional pre-loaded DataFrame.

        Parameters
        ----------
        df : pd.DataFrame, optional
            If provided, this DataFrame is used as the working dataset.
            If None, call `upload_data()` or `load_csv()` to load data.
        """
        self.df: pd.DataFrame = df.copy() if df is not None else None
        self._original_df: pd.DataFrame = df.copy() if df is not None else None

    def upload_data(self) -> None:
        """
        Trigger Google Colab's file upload dialog, read the uploaded CSV,
        and run automatic sanitisation.

        Falls back to a manual path prompt when not running inside Colab.
        """
        if IN_COLAB:
            uploaded = colab_files.upload()
            if not uploaded:
                print('No file uploaded.')
                return
            name, content = next(iter(uploaded.items()))
            self.df = pd.read_csv(io.BytesIO(content), na_values=self.GARBAGE_STRINGS)
        else:
            path = input('Enter path to CSV file: ').strip()
            self.df = pd.read_csv(path, na_values=self.GARBAGE_STRINGS)

        self._original_df = self.df.copy()
        self._auto_type_correction()
        print(f'Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns.')

    def load_csv(self, path: str) -> None:
        """
        Load a CSV from a local or URL path programmatically.

        Parameters
        ----------
        path : str — file path or URL to the CSV.
        """
        self.df = pd.read_csv(path, na_values=self.GARBAGE_STRINGS)
        self._original_df = self.df.copy()
        self._auto_type_correction()
        print(f'Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns.')

    def _auto_type_correction(self) -> None:
        """
        Attempt to coerce every object column to numeric.
        If the coercion would make the whole column NaN it is left as-is.
        """
        for col in self.df.select_dtypes(include='object').columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if converted.notna().any():
                self.df[col] = converted

    def reset(self) -> None:
        """Restore the DataFrame to its original loaded state."""
        if self._original_df is not None:
            self.df = self._original_df.copy()
            print('DataFrame reset to original state.')
        else:
            print('No original data to restore.')

    # 2 ── STRUCTURAL ANALYSIS & CLEANING
    #######################################################

    def _check_data(self) -> bool:
        """Return True if `self.df` is populated; print a warning otherwise."""
        if self.df is None or self.df.empty:
            print('No data loaded. Call upload_data() or load_csv() first.')
            return False
        return True

    def summary(self) -> None:
        """
        Print a structured summary of the dataset:
          - Shape (rows × columns)
          - Numerical vs categorical column breakdown
          - Missing-value counts per column
          - First 20 rows preview
        """
        if not self._check_data():
            return

        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=np.number).columns.tolist()

        print('=' * 60)
        print(f'  DATASET SUMMARY')
        print('=' * 60)
        print(f'  Rows      : {self.df.shape[0]}')
        print(f'  Columns   : {self.df.shape[1]}')
        print(f'  Numeric   : {len(num_cols)} → {num_cols}')
        print(f'  Categorical: {len(cat_cols)} → {cat_cols}')
        print()

        missing = self.df.isnull().sum()
        missing = missing[missing > 0]
        if missing.empty:
            print('  No missing values found.')
        else:
            print('  Missing Values:')
            for col, cnt in missing.items():
                pct = cnt / len(self.df) * 100
                print(f'    {col:30s}  {cnt:5d}  ({pct:.1f}%)')

        print()
        print('  First 20 rows:')
        display(self.df.head(20))
        print('=' * 60)

    def handle_missing_values(
        self,
        strategy: str = 'mean',
        constant=0,
        columns: list = None
    ) -> None:
        """
        Impute missing values using one of four strategies.

        Parameters
        ----------
        strategy : str
            One of ``'mean'``, ``'median'``, ``'mode'``, ``'constant'``.
            'mean' and 'median' apply only to numeric columns;
            'mode' applies to all dtypes.
        constant : scalar
            Fill value used when strategy == 'constant'.
        columns : list of str, optional
            Subset of columns to impute. Defaults to all columns.
        """
        if not self._check_data():
            return

        targets = columns if columns else self.df.columns.tolist()
        strategies = ('mean', 'median', 'mode', 'constant')
        if strategy not in strategies:
            print(f'Unknown strategy "{strategy}". Choose from {strategies}.')
            return

        for col in targets:
            if col not in self.df.columns:
                print(f'  Column "{col}" not found — skipped.')
                continue

            if self.df[col].isnull().sum() == 0:
                continue

            is_numeric = pd.api.types.is_numeric_dtype(self.df[col])

            if strategy == 'mean':
                if is_numeric:
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
                else:
                    print(f'  Skipping non-numeric column "{col}" for mean imputation.')

            elif strategy == 'median':
                if is_numeric:
                    self.df[col].fillna(self.df[col].median(), inplace=True)
                else:
                    print(f'  Skipping non-numeric column "{col}" for median imputation.')

            elif strategy == 'mode':
                mode_val = self.df[col].mode()
                if not mode_val.empty:
                    self.df[col].fillna(mode_val[0], inplace=True)

            elif strategy == 'constant':
                self.df[col].fillna(constant, inplace=True)

        remaining = self.df.isnull().sum().sum()
        print(f'Imputation done ({strategy}). Remaining NaNs: {remaining}.')

    def remove_duplicates(self) -> None:
        """
        Remove exact duplicate rows from the DataFrame.
        Reports how many duplicates were dropped.
        """
        if not self._check_data():
            return
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        dropped = before - len(self.df)
        print(f'Removed {dropped} duplicate row(s). Remaining rows: {len(self.df)}.')

    def handle_outliers(
        self,
        columns: list = None,
        action: str = 'flag',
        iqr_multiplier: float = 1.5
    ) -> None:
        """
        Detect outliers using the IQR method.

        Parameters
        ----------
        columns : list of str, optional
            Numeric columns to check. Defaults to all numeric columns.
        action : str
            ``'flag'``  — add a boolean column ``<col>_outlier`` per column.
            ``'delete'``— remove rows where ANY specified column is an outlier.
        iqr_multiplier : float
            Fence multiplier (default 1.5; use 3.0 for extreme outliers).
        """
        if not self._check_data():
            return

        num_cols = self.df.select_dtypes(include=np.number).columns.tolist()
        targets = [c for c in (columns or num_cols) if c in num_cols]

        if not targets:
            print('No valid numeric columns specified.')
            return

        outlier_mask = pd.Series(False, index=self.df.index)

        for col in targets:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - iqr_multiplier * IQR
            upper = Q3 + iqr_multiplier * IQR
            col_mask = (self.df[col] < lower) | (self.df[col] > upper)
            print(f'  {col}: {col_mask.sum()} outlier(s) [bounds: {lower:.2f}, {upper:.2f}]')

            if action == 'flag':
                self.df[f'{col}_outlier'] = col_mask
            outlier_mask |= col_mask

        if action == 'delete':
            before = len(self.df)
            self.df = self.df[~outlier_mask].reset_index(drop=True)
            print(f'Deleted {before - len(self.df)} outlier row(s). Remaining: {len(self.df)}.')

    def delete_rows(self) -> None:
        """
        Interactively delete rows by index.
        Prompts the user for a comma-separated list of row indices to drop.
        """
        if not self._check_data():
            return
        raw = input('Enter row indices to delete (comma-separated): ').strip()
        if not raw:
            print('No input provided.')
            return
        try:
            indices = [int(x.strip()) for x in raw.split(',')]
        except ValueError:
            print('Invalid input — please enter integer row indices.')
            return
        valid = [i for i in indices if i in self.df.index]
        invalid = [i for i in indices if i not in self.df.index]
        if invalid:
            print(f'  Indices not found and skipped: {invalid}')
        self.df.drop(index=valid, inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        print(f'Deleted {len(valid)} row(s). Remaining rows: {len(self.df)}.')

    def delete_columns(self) -> None:
        """
        Interactively delete columns by name.
        Prompts the user for a comma-separated list of column names to drop.
        """
        if not self._check_data():
            return
        raw = input('Enter column names to delete (comma-separated): ').strip()
        if not raw:
            print('No input provided.')
            return
        cols = [c.strip() for c in raw.split(',')]
        valid = [c for c in cols if c in self.df.columns]
        invalid = [c for c in cols if c not in self.df.columns]
        if invalid:
            print(f'  Columns not found and skipped: {invalid}')
        self.df.drop(columns=valid, inplace=True)
        print(f'Deleted columns: {valid}. Remaining columns: {list(self.df.columns)}.')


    # 3 ─NORMALISATION
    ##########################################################

    def extract_normalized_numeric_data(
        self,
        method: str = 'minmax',
        columns: list = None
    ) -> pd.DataFrame:
        """
        Scale numeric columns and return a new DataFrame.

        Parameters
        ----------
        method : str
            Scaling method: ``'minmax'`` (0–1), ``'standard'`` (Z-score),
            or ``'robust'`` (IQR-based, outlier-resistant).
        columns : list of str, optional
            Columns to scale. Defaults to all numeric columns.

        Returns
        -------
        pd.DataFrame — scaled numeric DataFrame.
        """
        if not self._check_data():
            return pd.DataFrame()

        scalers = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }
        if method not in scalers:
            print(f'Unknown method "{method}". Choose from {list(scalers.keys())}.')
            return pd.DataFrame()

        num_df = self.df.select_dtypes(include=np.number)
        if columns:
            valid_cols = [c for c in columns if c in num_df.columns]
            num_df = num_df[valid_cols]

        # Drop columns that are entirely NaN before scaling
        num_df = num_df.dropna(axis=1, how='all')
        # Fill remaining NaNs with column median for scaling purposes
        num_df = num_df.fillna(num_df.median())

        scaled = scalers[method].fit_transform(num_df)
        scaled_df = pd.DataFrame(scaled, columns=num_df.columns, index=num_df.index)
        print(f'Numeric data scaled using "{method}" method. Shape: {scaled_df.shape}')
        return scaled_df

    def extract_normalized_categorical_data(
        self,
        method: str = 'onehot',
        columns: list = None
    ) -> pd.DataFrame:
        """
        Encode categorical columns and return a new DataFrame.

        Parameters
        ----------
        method : str
            Encoding strategy:
            ``'onehot'``  — one-hot (dummy) encoding.
            ``'ordinal'`` — integer label encoding.
            ``'uniform'`` — ordinal integers scaled linearly to [0, 1].
        columns : list of str, optional
            Columns to encode. Defaults to all object/categorical columns.

        Returns
        -------
        pd.DataFrame — encoded categorical DataFrame.
        """
        if not self._check_data():
            return pd.DataFrame()

        cat_df = self.df.select_dtypes(exclude=np.number).copy()
        if columns:
            valid_cols = [c for c in columns if c in cat_df.columns]
            cat_df = cat_df[valid_cols]

        if cat_df.empty:
            print('No categorical columns found.')
            return pd.DataFrame()

        # Fill NaN with the string 'Unknown' before encoding
        cat_df = cat_df.fillna('Unknown')

        if method == 'onehot':
            encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            encoded = encoder.fit_transform(cat_df)
            feature_names = encoder.get_feature_names_out(cat_df.columns)
            result = pd.DataFrame(encoded, columns=feature_names, index=cat_df.index)

        elif method in ('ordinal', 'uniform'):
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoded = encoder.fit_transform(cat_df)
            result = pd.DataFrame(encoded, columns=cat_df.columns, index=cat_df.index)
            if method == 'uniform':
                result = result.apply(
                    lambda col: (col - col.min()) / (col.max() - col.min())
                    if col.max() != col.min() else col * 0
                )
        else:
            print(f'Unknown method "{method}". Choose from onehot, ordinal, uniform.')
            return pd.DataFrame()

        print(f'Categorical data encoded using "{method}" method. Shape: {result.shape}')
        return result

    def get_unified_dataset(
        self,
        numeric_method: str = 'minmax',
        categorical_method: str = 'onehot'
    ) -> pd.DataFrame:
        """
        Merge scaled numeric data and encoded categorical data into one DataFrame.

        Parameters
        ----------
        numeric_method     : str — passed to extract_normalized_numeric_data.
        categorical_method : str — passed to extract_normalized_categorical_data.

        Returns
        -------
        pd.DataFrame — unified feature DataFrame ready for ML pipelines.
        """
        num_data = self.extract_normalized_numeric_data(method=numeric_method)
        cat_data = self.extract_normalized_categorical_data(method=categorical_method)

        if num_data.empty and cat_data.empty:
            print('No data to merge.')
            return pd.DataFrame()

        unified = pd.concat([num_data, cat_data], axis=1)
        print(f'Unified dataset shape: {unified.shape}')
        return unified

    # 4 ─ INTERACTIVE VISUALISATION(PLOTLY)
    #########################################################

    def plot_univariate(self, column: str) -> None:
        """
        Generate a 3-panel subplot for a single numeric column:
          Panel 1 — Horizontal Box/Violin (distribution shape & spread)
          Panel 2 — Scatter plot (Index vs Value, trend over rows)
          Panel 3 — Histogram (frequency distribution)

        Parameters
        ----------
        column : str — name of a numeric column in self.df.
        """
        if not self._check_data():
            return
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return
        if not pd.api.types.is_numeric_dtype(self.df[column]):
            print(f'Column "{column}" is not numeric.')
            return

        series = self.df[column].dropna()

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=(
                f'Box / Violin — {column}',
                f'Scatter — {column}',
                f'Histogram — {column}'
            )
        )

        # Panel 1 — Violin + Box combined
        fig.add_trace(
            go.Violin(
                x=series, name=column,
                box_visible=True, meanline_visible=True,
                orientation='h', fillcolor='#636EFA',
                line_color='darkblue', opacity=0.6
            ),
            row=1, col=1
        )

        # Panel 2 — Scatter (row index vs value)
        fig.add_trace(
            go.Scatter(
                x=series.index, y=series,
                mode='markers',
                marker=dict(color='#EF553B', size=4, opacity=0.6),
                name=column
            ),
            row=1, col=2
        )

        # Panel 3 — Histogram
        fig.add_trace(
            go.Histogram(
                x=series, nbinsx=30,
                marker_color='#00CC96', opacity=0.8,
                name=column
            ),
            row=1, col=3
        )

        fig.update_layout(
            title=f'Univariate Analysis — {column}',
            height=400,
            showlegend=False,
            template='plotly_white'
        )
        fig.show()

    def plot_relationship(self, col_x: str, col_y: str) -> None:
        """
        Intelligently choose the correct chart based on column dtypes.

          Num–Num  → Scatter with OLS trendline.
          Cat–Num  → Box plot (category as x) with jittered points overlay.
          Cat–Cat  → Grouped / stacked bar chart of cross-tabulation.

        Parameters
        ----------
        col_x : str — name of the first column (x-axis / grouping variable).
        col_y : str — name of the second column (y-axis / target variable).
        """
        if not self._check_data():
            return
        for c in (col_x, col_y):
            if c not in self.df.columns:
                print(f'Column "{c}" not found.')
                return

        x_num = pd.api.types.is_numeric_dtype(self.df[col_x])
        y_num = pd.api.types.is_numeric_dtype(self.df[col_y])

        if x_num and y_num:
            # ── Num–Num: scatter + OLS trendline
            fig = px.scatter(
                self.df, x=col_x, y=col_y,
                trendline='ols',
                title=f'Scatter: {col_x} vs {col_y}',
                template='plotly_white',
                opacity=0.6
            )

        elif (not x_num and y_num) or (x_num and not y_num):
            # ── Cat–Num: box plot with all data point
            cat_col = col_x if not x_num else col_y
            num_col = col_y if not x_num else col_x
            fig = px.box(
                self.df, x=cat_col, y=num_col,
                points='all',
                title=f'Box Plot: {num_col} by {cat_col}',
                template='plotly_white'
            )

        else:
            # ── Cat–Cat: grouped bar of cross-tab counts
            ct = pd.crosstab(self.df[col_x], self.df[col_y]).reset_index()
            ct_melted = ct.melt(id_vars=col_x, var_name=col_y, value_name='Count')
            fig = px.bar(
                ct_melted, x=col_x, y='Count', color=col_y,
                barmode='group',
                title=f'Grouped Bar: {col_x} × {col_y}',
                template='plotly_white'
            )

        fig.show()

    def plot_categorical_frequency(self, column: str, top_n: int = 20) -> None:
        """
        Plot a bar chart of category frequencies with count and percentage labels.

        Parameters
        ----------
        column : str — name of a categorical column.
        top_n  : int — display only the top-N categories (default 20).
        """
        if not self._check_data():
            return
        if column not in self.df.columns:
            print(f'Column "{column}" not found.')
            return

        counts = self.df[column].value_counts().head(top_n)
        pcts = (counts / counts.sum() * 100).round(1)
        labels = [f'{c}  ({p}%)' for c, p in zip(counts.values, pcts.values)]

        fig = go.Figure(
            go.Bar(
                x=counts.index.astype(str),
                y=counts.values,
                text=labels,
                textposition='outside',
                marker_color='#AB63FA'
            )
        )
        fig.update_layout(
            title=f'Frequency — {column} (top {top_n})',
            xaxis_title=column,
            yaxis_title='Count',
            template='plotly_white',
            height=450
        )
        fig.show()

    # 5 ── DEEP STATISTICAL INSIGHTS
    ################################

    @staticmethod
    def _cramers_v(x: pd.Series, y: pd.Series) -> float:
        """
        Compute Cramér's V association between two categorical series.

        Parameters
        ----------
        x, y : pd.Series — categorical columns.

        Returns
        -------
        float — Cramér's V in [0, 1].
        """
        ct = pd.crosstab(x, y)
        n = ct.values.sum()
        chi2 = stats.chi2_contingency(ct, correction=False)[0]
        k = min(ct.shape) - 1
        if n == 0 or k == 0:
            return 0.0
        return float(np.sqrt(chi2 / (n * k)))

    @staticmethod
    def _eta_squared(cat: pd.Series, num: pd.Series) -> float:
        """
        Compute Eta² (effect size from one-way ANOVA) between a categorical
        and a numeric series.

        Parameters
        ----------
        cat : pd.Series — categorical grouping variable.
        num : pd.Series — numeric response variable.

        Returns
        -------
        float — Eta² in [0, 1].
        """
        groups = [grp.values for _, grp in num.groupby(cat)]
        if len(groups) < 2:
            return 0.0
        grand_mean = num.mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
        ss_total = ((num - grand_mean) ** 2).sum()
        return float(ss_between / ss_total) if ss_total != 0 else 0.0

    def plot_all_associations_heatmap(self, max_cols: int = 20) -> None:
        """
        Compute and visualise a unified association matrix across ALL columns:

          Num–Num  → Pearson's r  (linear correlation)
          Cat–Cat  → Cramér's V   (chi-square based)
          Num–Cat  → Eta²         (ANOVA-based effect size)

        Parameters
        ----------
        max_cols : int — limit to first N columns to keep chart readable.
        """
        if not self._check_data():
            return

        # Use only non-all-NaN columns, capped at max_cols
        cols = [c for c in self.df.columns if self.df[c].notna().any()][:max_cols]
        n = len(cols)
        matrix = np.zeros((n, n))

        df = self.df[cols].copy()

        for i, ci in enumerate(cols):
            for j, cj in enumerate(cols):
                if i == j:
                    matrix[i][j] = 1.0
                    continue
                if j < i:           # Mirror lower triangle
                    matrix[i][j] = matrix[j][i]
                    continue

                xi = df[ci].dropna()
                xj = df[cj].dropna()
                common = xi.index.intersection(xj.index)
                xi, xj = xi[common], xj[common]

                if len(common) < 5:
                    matrix[i][j] = 0.0
                    continue

                i_num = pd.api.types.is_numeric_dtype(df[ci])
                j_num = pd.api.types.is_numeric_dtype(df[cj])

                try:
                    if i_num and j_num:
                        r, _ = stats.pearsonr(xi, xj)
                        matrix[i][j] = abs(r)
                    elif not i_num and not j_num:
                        matrix[i][j] = self._cramers_v(xi, xj)
                    else:
                        num_s = xi if i_num else xj
                        cat_s = xj if i_num else xi
                        matrix[i][j] = self._eta_squared(cat_s, num_s)
                except Exception:
                    matrix[i][j] = 0.0

        assoc_df = pd.DataFrame(matrix, index=cols, columns=cols)

        fig = px.imshow(
            assoc_df,
            color_continuous_scale='RdBu_r',
            zmin=0, zmax=1,
            title='Unified Association Heatmap (Pearson r | Cramér V | Eta²)',
            text_auto='.2f',
            aspect='auto',
            template='plotly_white'
        )
        fig.update_layout(height=600, width=800)
        fig.show()

print('DataInspector class defined.')

## Step 4 — Real-World Demonstration (Titanic Dataset)

In [ ]:
# ── 4.1  Load the Titanic dataset from a public URL
inspector = DataInspector()
inspector.load_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

In [ ]:
# ── 4.2  Structural summary
inspector.summary()

In [ ]:
# ── 4.3  Remove duplicates
inspector.remove_duplicates()

In [ ]:
# ── 4.4  Intelligent imputation
# Age → median  (numeric, skewed)
# Embarked → mode  (categorical)
# Fare → median
inspector.handle_missing_values(strategy='median', columns=['Age', 'Fare'])
inspector.handle_missing_values(strategy='mode',   columns=['Embarked'])

In [ ]:
# ── 4.5  Flag IQR outliers on Fare
inspector.handle_outliers(columns=['Fare'], action='flag')

In [ ]:
# ── 4.6  Univariate analysis
pector.plot_univariate('Age')
inspector.plot_univariate('Fare')

In [ ]:
# ── 4.7  Relationship plots
# Num–Num  ─ Age vs Fare (scatter + OLS)
inspector.plot_relationship('Age', 'Fare')

# Cat–Num  ─ Pclass vs Fare (box + points)
inspector.plot_relationship('Pclass', 'Fare')

# Cat–Cat  ─ Sex vs Survived (grouped bar)
inspector.plot_relationship('Sex', 'Survived')

In [ ]:
# ── 4.8  Categorical frequency
inspector.plot_categorical_frequency('Embarked')
inspector.plot_categorical_frequency('Pclass')

In [ ]:
# ── 4.9  Full association heatmap
inspector.plot_all_associations_heatmap()

In [ ]:
# ── 4.10  Normalisation & unified dataset

# Drop columns not needed for feature engineering
inspector.df.drop(columns=['Name', 'Ticket', 'Cabin', 'Fare_outlier'], errors='ignore', inplace=True)

# Build unified scaled + encoded dataset
unified_df = inspector.get_unified_dataset(
    numeric_method='standard',
    categorical_method='onehot'
)
print('Unified dataset preview:')
display(unified_df.head())

In [ ]:
# ── 4.11  Demonstrate PlottingMethods standalone class
pm = PlottingMethods()

# Bar chart — survival counts
surv_counts = inspector.df['Survived'].value_counts()
pm.bar_chart(
    categories=['Did Not Survive', 'Survived'],
    values=surv_counts.values,
    title='Titanic — Survival Counts',
    y_label='Count'
)

# Pie chart — passenger class distribution
pclass_counts = inspector.df['Pclass'].value_counts()
pm.pie_chart(
    labels=[f'Class {c}' for c in pclass_counts.index],
    values=pclass_counts.values,
    title='Titanic — Passenger Class Distribution'
)

# Histogram — age distribution
pm.histogram(inspector.df, column='Age', nbins=25, title='Titanic — Age Distribution')